In [22]:
import pandas as pd
import numpy as np
import sys
sys.path.append("C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code/")
from bmresearch_tools import BMR_load, BMR_plot, get_metadata
%matplotlib widget
import matplotlib.pyplot as plt
import datetime
import h5py
import os

In [105]:
studyID = '002'
ecg_original_path = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_studyID/BMR_002.h5'
resampled_ecg_path = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG/ECG_002.h5'
nn_output_toolbox = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/HRV_analysis/002_NNIntervals.csv'
nn_transformed_path = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/HRV_analysis/002_NN_RPeaks.csv'

In [3]:
lead = 'I'
with h5py.File(resampled_ecg_path,'r') as f:
#     print(f.keys())
    start_datetime = f[lead + '_startdatetime'][:]

In [4]:
start_datetime

array([2018,    6,   11,   14,   27,    0,    0], dtype=int64)

In [5]:
start_datetime = [str(x) for x in start_datetime]
start_datetime = start_datetime[0] + '-' + start_datetime[1] + '-' + start_datetime[2] + ' ' +start_datetime[3]+':'+start_datetime[4]+':'+start_datetime[5]+'.'+start_datetime[6]
start_datetime = pd.to_datetime(start_datetime, infer_datetime_format=1)

In [6]:
start_datetime

Timestamp('2018-06-11 14:27:00')

In [7]:
# read in NN data after running cardiovascular toolbox:
nn = pd.read_csv(nn_output_toolbox)
nn.time

0         0.766667
1         1.308333
2         1.854167
3         2.400000
4         2.945833
           ...    
1527    897.783333
1528    898.229167
1529    898.679167
1530    899.120833
1531    899.570833
Name: time, Length: 1532, dtype: float64

In [32]:
nn['peak_index'] = np.int64(np.round(nn.time.values*240))

In [42]:
indices = nn.time.values*240
# np.int64(np.round(indices[1]))
indices[0]

184.00000000008

In [34]:
nn['peak_datetime'] = [start_datetime + datetime.timedelta(seconds=x) for x in nn.time.values]
# nn_datetime

In [35]:
nn

,time,nn,peak_index,peak_datetime
0,0.766667,0.545833,184,2018-06-11 14:27:00.766667
1,1.308333,0.541667,314,2018-06-11 14:27:01.308333
2,1.854167,0.545833,445,2018-06-11 14:27:01.854167
3,2.400000,0.545833,576,2018-06-11 14:27:02.400000
4,2.945833,0.545833,707,2018-06-11 14:27:02.945833
...,...,...,...,...
1527,897.783333,0.445833,215468,2018-06-11 14:41:57.783333
1528,898.229167,0.445833,215575,2018-06-11 14:41:58.229167
1529,898.679167,0.450000,215683,2018-06-11 14:41:58.679167
1530,899.120833,0.441667,215789,2018-06-11 14:41:59.120833


In [36]:
nn.to_csv(nn_transformed_path, index = False)

In [78]:
ecg_resampled_data = h5py.File(resampled_ecg_path, 'r')
print(ecg_resampled_data.keys())
ecg_resampled_data = ecg_resampled_data['I'][:] # [:20000]


<KeysViewHDF5 ['I', 'I_startdatetime']>


In [79]:
ecg_resampled_data

array([-1.1000e+01,  7.0000e+00, -5.0000e+00, ..., -3.2752e+04,
       -3.2752e+04, -3.2752e+04])

In [102]:
# do peak correction (possibly due to small errors/rounding erros before in scripts. check for max value in neighboring values):
nn['peak_index'] = [index + np.argmax(ecg_resampled_data[index-1:index+2])-1 for index in nn['peak_index'].values]

In [103]:
plt.close()
xaxis = np.arange(ecg_resampled_data.shape[0])
plt.plot(xaxis[:20000], ecg_resampled_data[:20000])
plt.scatter(xaxis[nn['peak_index'].values[:200]], ecg_resampled_data[nn['peak_index'].values[:200]],c='r')
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [106]:
ecg_original = BMR_load(ecg_original_path, signals ='I')
ecg_original = ecg_original['I']

In [107]:
ecg_original.shape

(27250756, 3)

In [108]:
nn.peak_datetime.shape[0]

1532

In [136]:
import time
start_time = time.time()
# first idx_min
idx_min = abs(ecg_original.datetime - nn.peak_datetime[0]).idxmin()
rr_idx = [idx_min]
for x in nn.peak_datetime[1:]:
    idx_min = abs(ecg_original.datetime.iloc[idx_min:idx_min+36000] - x).idxmin()
    rr_idx.append(idx_min)
print( time.time() - start_time)
    

2.358610153198242


In [137]:
# do peak correction (possibly due to small errors/rounding erros before in scripts. check for max value in neighboring values):
rr_idx = [index + ecg_original.signal.iloc[index-15:index+15].values.argmax()-15 for index in rr_idx]

In [138]:
# ecg_dt_peak1 = np.array(ecg_dt_peak) - 1

In [139]:
plt.close()
plt.plot(ecg_original.datetime[:1000000], ecg_original.signal[:1000000])
plt.scatter(ecg_original.datetime[rr_idx], ecg_original.signal[rr_idx],c='r')
# plt.scatter(xaxis[nn['peak_index'].values[:80]], ecg_resampled_data[nn['peak_index'].values[:80]],c='r')
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …